# LILaC reproduction (MMCoQA) — Colab driver
Runs `scripts/lilac/pipeline.sh` stage by stage. *Runtime → Run all* is safe: every cell stops at the first failed command, every stage is resumable, and results are copied to Google Drive after each stage.

### 0. Session setup
Runtime → *Change runtime type* → **L4 GPU** (Colab Pro, recommended: every model of the official pipeline fits in 24 GB, about 1.7 compute units per hour). On an A100 the 32B decomposer is added. A 16 GB T4 also works: 4-bit AWQ checkpoints for Qwen2.5-VL and the decomposers and 8-bit MM-Embed are then selected automatically.

Keep this browser tab open while it runs (Colab Pro has no background execution). If the session ends, open the notebook again and *Run all*: finished work is restored from Drive and skipped.

Store `WANDB_API_KEY` and `HF_TOKEN` in Colab *Secrets* (key icon on the left) and enable *Notebook access* for both. Never paste keys into the notebook: the repository is public.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
for k in ('WANDB_API_KEY', 'HF_TOKEN'):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print('secret not set:', k)
# W&B runs go to the default entity of WANDB_API_KEY; set WANDB_ENTITY to use a team instead.
os.environ['ENV_ROOT'] = '/content/envs'

import codecs, signal, subprocess, sys

def sh(cmd, check=True):
    """Run a shell command and stream its output. A failing command raises an error,
    so 'Run all' stops at the first failed stage instead of running the next ones."""
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, start_new_session=True)
    dec = codecs.getincrementaldecoder('utf-8')('replace')
    try:
        while True:
            b = os.read(p.stdout.fileno(), 4096)
            if not b:
                break
            sys.stdout.write(dec.decode(b)); sys.stdout.flush()
        rc = p.wait()
    except KeyboardInterrupt:          # 'Interrupt execution' also stops the GPU job
        os.killpg(p.pid, signal.SIGINT)
        try:
            p.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid, signal.SIGKILL)
        raise
    if rc != 0 and check:
        raise RuntimeError(f'command failed (exit code {rc}): {cmd}')

In [ ]:
%cd /content
sh('git clone https://github.com/NoiceSmoker/mmgraphrag-repro.git 2>/dev/null || (cd mmgraphrag-repro && git pull)')
%cd /content/mmgraphrag-repro
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
                     capture_output=True, text=True)
if smi.returncode != 0:
    raise RuntimeError('No GPU in this session: Runtime -> Change runtime type -> L4 GPU')
gpu_name, gpu_mib = smi.stdout.strip().split('\n')[0].rsplit(', ', 1)
gpu_mib = int(float(gpu_mib))
if gpu_mib < 20000:      # T4 (16 GB): 4-bit VLM / decomposers, 8-bit MM-Embed, fp16
    os.environ['VLM_MODEL'] = 'Qwen/Qwen2.5-VL-7B-Instruct-AWQ'
    os.environ['DECOMPOSERS'] = ('qwen2.5-7b-awq=Qwen/Qwen2.5-7B-Instruct-AWQ '
                                 'qwen2.5-14b-awq=Qwen/Qwen2.5-14B-Instruct-AWQ')
    os.environ['MMEMBED_QUANT'] = '8bit'
    os.environ['MMEMBED_BATCH_SCALE'] = '0.5'
    mode = 'T4 mode (quantized models)'
elif gpu_mib < 30000:    # L4 (24 GB): official models; the 32B decomposer does not fit
    os.environ['DECOMPOSERS'] = ('qwen2.5-7b=Qwen/Qwen2.5-7B-Instruct '
                                 'qwen2.5-14b-awq=Qwen/Qwen2.5-14B-Instruct-AWQ')
    mode = 'L4 mode (official models, decomposers 7B + 14B-AWQ)'
else:                    # A100 / H100: official models, all three decomposers
    mode = 'large-GPU mode (official models, decomposers 7B + 14B-AWQ + 32B-AWQ)'
print(f'{gpu_name} ({gpu_mib} MiB) -> {mode}')

### 1. Environments (once per VM, ~5–10 min)

In [ ]:
sh('bash scripts/setup_envs.sh lilac-base mmembed vllm')

### 2. Official code + data (restores earlier artefacts from Drive if present)

In [ ]:
sh('bash scripts/colab/sync.sh pull lilac')
sh('bash scripts/lilac/pipeline.sh fetch data && bash scripts/colab/sync.sh push lilac')

### 3. Graph construction: Qwen2.5-VL passes (vLLM) + official steps 1–7

In [ ]:
sh('bash scripts/lilac/pipeline.sh vlm && bash scripts/colab/sync.sh push lilac')
sh('bash scripts/lilac/pipeline.sh graph && bash scripts/colab/sync.sh push lilac')

### 4. MM-Embed corpus embeddings (resumable per component)

In [ ]:
sh('bash scripts/lilac/pipeline.sh embed && bash scripts/colab/sync.sh push lilac')

### 5. Query decomposition with several LLMs + sub-query embeddings
`decompose_llm` writes the sub-queries of every decomposer, `decompose_embed` embeds them with the same MM-Embed setting as step 4.

In [ ]:
sh('bash scripts/lilac/pipeline.sh decompose_llm && bash scripts/colab/sync.sh push lilac')
sh('bash scripts/lilac/pipeline.sh decompose_embed && bash scripts/colab/sync.sh push lilac')

### 6. Extra ablation: image nodes without VLM text

In [ ]:
sh('bash scripts/lilac/pipeline.sh novlm && bash scripts/colab/sync.sh push lilac')

### 7. Retrieval experiments → W&B (CPU work)

In [ ]:
sh('bash scripts/lilac/pipeline.sh experiments && bash scripts/colab/sync.sh push lilac')

### 8. W&B report pages + LaTeX tables for the written report
Links of both papers are collected in `results/wandb_links.json` on Drive; the LaTeX fragments in `results/latex` are complete once both notebooks have finished.

In [ ]:
sh('bash scripts/colab/sync.sh pull results')
sh('/content/envs/lilac-base/bin/python scripts/make_wandb_reports.py --only lilac --share')
sh('/content/envs/lilac-base/bin/python scripts/export_results.py --results results --out results/latex')
sh('bash scripts/colab/sync.sh push results')